# Arkansas assignment experiments

Run the selected periods, save each assignment through the project results manager and compare it with an available reference run. Use Python 3.12 with AequilibraE 2.0 and SpatiaLite.

In [ ]:
import json
import os
from pathlib import Path
import sys

os.environ["AEQ_SHOW_PROGRESS"] = "FALSE"

import pandas as pd
from aequilibrae import Project
from IPython.display import display

src_dir = Path("src")
if not src_dir.is_dir():
    src_dir = Path("benchmarks/arkansas/src")
if src_dir.is_dir():
    sys.path.insert(0, str(src_dir.resolve()))

from assignments import assignment
from build_graphs import build_network
from format_matrices import format_matrices

model_path = Path("/mnt/d/models/Arkansas/model")
periods = ["am"]
result_suffix = "EXPERIMENT"

In [ ]:
with Project(model_path) as project:
    format_matrices(project)
    network = build_network(project, periods)
    runs = assignment(project, network, periods)
    for period, run in runs.items():
        run.save_results(f"{period.upper()}_{result_suffix}")

runs[periods[0]].report()

In [ ]:
comparisons = {}
report_names = [f"{period.upper()}_{result_suffix}" for period in periods]
with Project(model_path) as project:
    metadata = project.results.list()
    saved_names = set(metadata["table_name"])
    for period, run in runs.items():
        reference = f"{period.upper()}_FULLY_CONVERGED"
        if reference not in saved_names and period == "am":
            reference = "AM_w_turn_penalties_EXPERIMENT"
        if reference not in saved_names:
            print(f"No saved reference for {period.upper()}.")
            continue

        report_names.append(reference)
        saved = project.results.get_results(reference).set_index("link_id")["PCE_tot"]
        comparison = pd.concat(
            [saved.rename("PCE_REFERENCE"), run.results()["PCE_tot"].rename("PCE_EXPERIMENT")],
            axis=1,
            join="inner",
        )
        comparisons[period] = comparison
        ax = comparison.plot.scatter(x="PCE_REFERENCE", y="PCE_EXPERIMENT")
        ax.set_title(f"{period.upper()} vs {reference}")
        ax.figure.savefig(model_path / f"{period}_experiment_vs_converged.png")

    reports = metadata.loc[metadata["table_name"].isin(report_names)].copy()

reports.to_csv(model_path / "all_results_report.csv", index=False)

In [ ]:
gaps = []
for record in reports.itertuples():
    report = json.loads(record.procedure_report)
    if isinstance(report, str):
        report = json.loads(report)
    convergence = pd.DataFrame(report["convergence"]).set_index("iteration")
    gaps.append(convergence["rgap"].rename(record.table_name))

convergence_all = pd.concat(gaps, axis=1).sort_index()
convergence_all.to_csv(model_path / "convergence_all.csv")

for period in periods:
    names = [name for name in convergence_all if name.startswith(f"{period.upper()}_")]
    positive_gaps = convergence_all[names].where(
        convergence_all[names].gt(0) & convergence_all[names].lt(float("inf"))
    )
    ax = positive_gaps.plot(logy=True, ylabel="Relative gap")
    ax.figure.savefig(model_path / f"convergence_{period}_log.png")

In [ ]:
rmse = pd.Series(
    {
        period: ((frame["PCE_EXPERIMENT"] - frame["PCE_REFERENCE"]) ** 2).mean() ** 0.5
        for period, frame in comparisons.items()
    },
    name="Flow RMSE",
)
display(rmse)